
# 09 · Recommendation audit

What do real people actually get, and which input causes what? A workbench to look at outcomes side by side before changing the app, so nothing in the ranking is a black box.

What is validated and what isn't:

| Part | Status |
|---|---|
| **Reach**: where people from a job actually move (LightGBM + CPS history) | validated: hit@10 0.62 on held-out 2022-25 moves (notebook 04) |
| Text placement (description / CV) | measured in notebook 07: helps on pseudo-CVs, weak on short real text |
| Skill answers (15 questions) | nudge the title's profile by 25%; survey: ~1.2 points noise |
| Interest fit (topics, quiz), goal multipliers, experience multiplier, the "best fit" blend | **design choices, not validated** |

Inputs: the 17 survey respondents (title, description, 15 skill answers) and Tim's own saved profile.


In [1]:

import sys, json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from api.engine import Engine, Profile

engine = Engine()
pd.set_option('display.max_colwidth', 40)



## Inputs

Survey respondents become profiles: the job title is Tim's assigned occupation (as if they picked it from the dropdown), plus their description and 15 skill answers.


In [2]:

survey = pd.read_csv(ROOT / 'data/raw/survey/responses.csv')
key_of = {item['name']: item['key'] for item in engine.meta['items']}
item_cols = {c: key_of[c.replace('Knowledge of ', '').replace('Skill: ', '')] for c in survey.columns[3:]}

TRUE_JOB = [110, 1107, 1020, 1030, 1600, 5860, 1860, 1020, 1610, 1860, 2010, 2000, 2100, 2100, 2010, 1860, 1030]

people = []
for i, row in survey.iterrows():
    description = row.iloc[2] if isinstance(row.iloc[2], str) and len(row.iloc[2].strip()) > 10 else None
    people.append({
        'who': f"#{i + 1} {row.iloc[1].strip()}",
        'job': TRUE_JOB[i],
        'answers': {item_cols[c]: float(row[c]) for c in item_cols if pd.notna(row[c])},
        'description': description,
    })

tim = Profile(**json.loads((ROOT / 'data/raw/survey/tim-profile.json').read_text())['profile'])          # before the CV
tim_cv = Profile(**json.loads((ROOT / 'data/raw/survey/tim-profile-cv.json').read_text())['profile'])   # after the CV
len(people), engine.census.at[tim.job, 'title']


(17, 'Computer and information systems managers')


## Looking inside one recommendation

`components(profile)` returns every candidate stop with the numbers behind its rank: reach (share of real moves and its rank), skill match, interest fit (`appeal_pct`), and the final blended `best`. Sort by any column to see what drives the order.


In [3]:

COLUMNS = ['title', 'reach_label', 'reach_rank', 'reach_share', 'skill_match', 'appeal_pct', 'median_wage', 'bls_ai_exposure', 'best']


def components(profile, n=15, sort='best'):
    rec = engine.recommend(profile, sort=sort, limit=600)
    frame = pd.DataFrame(rec['results']).set_index('census')
    return frame[[c for c in COLUMNS if c in frame]].head(n)


components(tim)


,title,reach_label,reach_rank,reach_share,skill_match,appeal_pct,median_wage,bls_ai_exposure,best
census,,,,,,,,,
50,Marketing and sales managers,Common move,1.0,0.2329,0.5124,0.2635,155555.6508,High,0.1332
430,"Compliance, project & other managers",Common move,3.0,0.0973,0.4910,0.3820,122763.0605,High,0.0680
120,Financial managers,Common move,2.0,0.1983,0.4798,0.1712,166570.0000,Very high,0.0620
100,Administrative services managers,Common move,4.0,0.0515,0.3418,0.3153,111344.4909,Moderate,0.0252
350,Medical and health services managers,Common move,7.0,0.0239,0.4445,0.3709,123860.0000,High,0.0178
40,Advertising and promotions managers,Common move,8.0,0.0209,0.3806,0.5097,133660.0000,Very high,0.0174
60,Public relations and fundraising man...,Common move,5.0,0.0345,0.2684,0.5522,138836.9136,Very high,0.0161
230,Education administrators,Common move,6.0,0.0240,0.4205,0.5687,99013.8397,High,0.0144
220,Construction managers,Occasional move,11.0,0.0155,0.2119,0.4526,114990.0000,High,0.0127



## Approaches

Each takes a profile and returns the top stops. Add your own.

- **current**: the app today (blended best fit)
- **natural next steps**: reach only (the validated part)
- **toward what you love**: interest fit only, among stops that are at least a rare-but-real move (reach rank ≤ 150)


In [4]:

def current(profile, n=6):
    return components(profile, n, 'best')['title'].tolist()


def natural_next_steps(profile, n=6):
    return components(profile, n, 'reachable')['title'].tolist()


def toward_what_you_love(profile, n=6, max_reach_rank=150):
    frame = components(profile, 600, 'appealing')
    if frame['appeal_pct'].isna().all():
        return []
    if frame['reach_rank'].notna().any():
        frame = frame[frame['reach_rank'] <= max_reach_rank]
    return frame['title'].head(n).tolist()


def top_moves_by_interest(profile, n=6, pool=20):
    """Real moves decide what's on the list (top `pool` by reach, the validated zone);
    interests only decide the order within it. Without interests: plain reach order."""
    frame = components(profile, 600, 'reachable').nsmallest(pool, 'reach_rank')
    if frame['appeal_pct'].notna().any():
        frame = frame.sort_values('appeal_pct', ascending=False)
    return frame['title'].head(n).tolist()


def core_moves_by_interest(profile, n=6, coverage=0.8, min_skill=0.0):
    """The pool is the moves that together make up `coverage` of all real moves from here (not a fixed 20),
    minus stops whose skills don't match at all (skill_match <= min_skill: physicians from tech, etc.).
    Interests only order the pool."""
    frame = components(profile, 600, 'reachable').sort_values('reach_rank')
    pool = frame[frame['reach_share'].cumsum().shift(fill_value=0) < coverage * frame['reach_share'].sum()]
    pool = pool[pool['skill_match'] > min_skill]
    if pool['appeal_pct'].notna().any():
        pool = pool.sort_values('appeal_pct', ascending=False)
    return pool['title'].head(n).tolist()


def moves_with_interest_nudge(profile, n=6, low=0.5):
    """Real moves decide; interests only nudge: score = move share x (low + interest fit).
    An interest match can lift a stop at most x1.5 and sink it to x0.5 (low=0.5), so it moves a few places,
    but a rare move can't jump a common one."""
    frame = components(profile, 600, 'reachable')
    nudge = (low + frame['appeal_pct']).fillna(1.0)
    return frame.assign(score=frame['reach_share'] * nudge).sort_values('score', ascending=False)['title'].head(n).tolist()


APPROACHES = {'current': current, 'natural next steps': natural_next_steps, 'toward what you love': toward_what_you_love,
              'top 20 moves, by interest': top_moves_by_interest,
              'core moves (80%, skills fit), by interest': core_moves_by_interest,
              'moves, interest nudge (app next)': moves_with_interest_nudge}

pd.DataFrame({name: fn(tim) for name, fn in APPROACHES.items()})


,current,natural next steps,toward what you love,"top 20 moves, by interest","core moves (80%, skills fit), by interest","moves, interest nudge (app next)"
0,Marketing and sales managers,Marketing and sales managers,Physical therapists,"Software developers, applications an...","Software developers, applications an...",Marketing and sales managers
1,"Compliance, project & other managers",Financial managers,"Architects, except naval",Computer network architects,Education administrators,Financial managers
2,Financial managers,"Compliance, project & other managers",Secondary school teachers,Education administrators,Public relations and fundraising man...,"Compliance, project & other managers"
3,Administrative services managers,Administrative services managers,Elementary and middle school teachers,Architectural and engineering managers,Lawyers,Administrative services managers
4,Medical and health services managers,Public relations and fundraising man...,Preschool and kindergarten teachers,Public relations and fundraising man...,Social and community service managers,Public relations and fundraising man...
5,Advertising and promotions managers,Education administrators,Clergy,Lawyers,Advertising and promotions managers,Education administrators


## Tim before and after the CV, every approach

Before: title = computer and information systems managers, 15 answers. After: title = computer and information research scientists + software developers, CV activities and tools, 6 years.

In [5]:
pd.concat({
    'before CV': pd.DataFrame({name: fn(tim) for name, fn in APPROACHES.items()}),
    'after CV': pd.DataFrame({name: fn(tim_cv) for name, fn in APPROACHES.items()}),
})

current  \
before CV 0          Marketing and sales managers   
          1  Compliance, project & other managers   
          2                    Financial managers   
          3      Administrative services managers   
          4  Medical and health services managers   
          5   Advertising and promotions managers   
after CV  0             Computer systems analysts   
          1                  Computer programmers   
          2          Computer support specialists   
          3                        Web developers   
          4          Computer network architects    
          5         Information security analysts   

                                  natural next steps  \
before CV 0             Marketing and sales managers   
          1                       Financial managers   
          2     Compliance, project & other managers   
          3         Administrative services managers   
          4  Public relations and fundraising man...   
          5                 Education administrators   
after CV  0                Computer systems analysts   
          1                     Computer programmers   
          2             Computer support specialists   
          3                           Web developers   
          4            Information security analysts   
          5             Operations research analysts   

                              toward what you love  \
before CV 0                    Physical therapists   
          1               Architects, except naval   
          2              Secondary school teachers   
          3  Elementary and middle school teachers   
          4    Preschool and kindergarten teachers   
          5                                 Clergy   
after CV  0               Architects, except naval   
          1                   Biomedical engineers   
          2                               Drafters   
          3                    Materials engineers   
          4              Secondary school teachers   
          5  Elementary and middle school teachers   

                           top 20 moves, by interest  \
before CV 0  Software developers, applications an...   
          1             Computer network architects    
          2                 Education administrators   
          3   Architectural and engineering managers   
          4  Public relations and fundraising man...   
          5                                  Lawyers   
after CV  0                 Architects, except naval   
          1                                Designers   
          2       Robotics, energy & other engineers   
          3  Remote sensing & other physical scie...   
          4                           Web developers   
          5  Environmental scientists and geoscie...   

            core moves (80%, skills fit), by interest  \
before CV 0  Software developers, applications an...    
          1                 Education administrators    
          2  Public relations and fundraising man...    
          3                                  Lawyers    
          4    Social and community service managers    
          5      Advertising and promotions managers    
after CV  0                                Designers    
          1       Robotics, energy & other engineers    
          2  Remote sensing & other physical scie...    
          3                           Web developers    
          4     Electrical and electronics engineers    
          5  Environmental scientists and geoscie...    

                    moves, interest nudge (app next)  
before CV 0             Marketing and sales managers  
          1                       Financial managers  
          2     Compliance, project & other managers  
          3         Administrative services managers  
          4  Public relations and fundraising man...  
          5                 Education administrators  
after CV  0                Computer systems analysts  
         


## Tim's profile: which input does what?

The same approach with one input switched off at a time.


In [6]:

ablations = {
    'as entered': tim,
    'no goals': replace(tim, goals=[]),
    'no interests (topics + quiz)': replace(tim, chips=[], quiz={}),
    'no skill answers': replace(tim, answers={}),
    'experience 7 years': replace(tim, experience_years=7),
    'title only': Profile(job=tim.job),
}
pd.DataFrame({name: current(p) for name, p in ablations.items()}).T


,0,1,2,3,4,5
as entered,Marketing and sales managers,"Compliance, project & other managers",Financial managers,Administrative services managers,Medical and health services managers,Advertising and promotions managers
no goals,Marketing and sales managers,Financial managers,"Compliance, project & other managers",Administrative services managers,Public relations and fundraising man...,"Software developers, applications an..."
no interests (topics + quiz),Marketing and sales managers,Financial managers,"Compliance, project & other managers",Administrative services managers,Medical and health services managers,Advertising and promotions managers
no skill answers,Marketing and sales managers,Financial managers,"Compliance, project & other managers",Administrative services managers,Advertising and promotions managers,Public relations and fundraising man...
experience 7 years,Marketing and sales managers,Financial managers,"Compliance, project & other managers",Public relations and fundraising man...,Administrative services managers,Education administrators
title only,Marketing and sales managers,Financial managers,"Compliance, project & other managers",Administrative services managers,Public relations and fundraising man...,Advertising and promotions managers



## The 17 respondents: what does each input add?

Same person, growing inputs. `answers only` is the questionnaire without a title: can the 15 questions place someone on their own?


In [7]:

def sweep(person, n=4):
    base = Profile(job=person['job'])
    rows = {
        'title only': natural_next_steps(base, n),
        '+ skill answers': natural_next_steps(replace(base, answers=person['answers']), n),
    }
    if person['description']:
        rows['+ description'] = natural_next_steps(replace(base, answers=person['answers'], description=person['description']), n)
    rows['answers only (no title)'] = natural_next_steps(Profile(answers=person['answers']), n)
    return rows


for person in people:
    print(person['who'], '->', engine.census.at[person['job'], 'title'])
    for name, tops in sweep(person).items():
        print(f'   {name:24s}', ' | '.join(t[:28] for t in tops))


#1 Engineering manager -> Computer and information systems managers
   title only               Marketing and sales managers | Financial managers | Compliance, project & other  | Administrative services mana
   + skill answers          Financial managers | Marketing and sales managers | Compliance, project & other  | Administrative services mana
   + description            Financial managers | Marketing and sales managers | Compliance, project & other  | Administrative services mana
   answers only (no title)  Nursing, psychiatric, and ho | Registered nurses | Medical assistants | Physicians and surgeons
#2 Product designer -> IT project managers, QA testers, UI designers & more tech


   title only               Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   + skill answers          Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   + description            Computer support specialists | Computer systems analysts | Network and computer systems | Computer programmers
   answers only (no title)  Artists and related workers | Producers and directors | Janitors and building cleane | Writers and authors
#3 Software engineer -> Software developers, applications and systems software
   title only               IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + skill answers          IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + description            Computer systems analysts | Computer support sp

   title only               Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   + skill answers          Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   answers only (no title)  Assemblers & fabricators | Inspectors, testers, sorters | Welding, soldering, and braz | First-line supervisors of pr
#5 Research Scientist -> Agricultural and food scientists
   title only               Medical scientists | Remote sensing & other physi | Psychologists | Biological scientists
   + skill answers          Medical scientists | Remote sensing & other physi | Psychologists | Biological scientists
   answers only (no title)  Remote sensing & other physi | Quality control, environment | Registered nurses | Medical scientists
#6 Office Coordinator -> Office clerks, general


   title only               Secretaries and administrati | Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers
   + skill answers          Secretaries and administrati | Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers
   + description            Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers | Receptionists and informatio
   answers only (no title)  Secretaries and administrati | Customer service representat | Childcare workers | Elementary and middle school
#7 Senior Researcher -> Planners, anthropologists, historians & more
   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + description            Psychologists | Remote sensing & other physi | Environmental scientists and | M

   title only               IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + skill answers          IT project managers, QA test | Computer programmers | Computer support specialists | Computer systems analysts
   + description            Computer support specialists | Computer programmers | Web developers | Network and computer systems
   answers only (no title)  Assemblers & fabricators | Automotive service technicia | Welding, soldering, and braz | CNC operators & other produc
#9 Postdoctoral researcher -> Biological scientists
   title only               Remote sensing & other physi | Medical scientists | Environmental scientists and | Chemists and materials scien
   + skill answers          Remote sensing & other physi | Medical scientists | Environmental scientists and | Chemists and materials scien
   answers only (no title)  Registered nurses | Nursing, psychiatric, and ho | Physicians and surgeons | Health prac

   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Environmental scientists and | Remote sensing & other physi
   + description            Psychologists | Urban and regional planners | Environmental scientists and | Remote sensing & other physi
   answers only (no title)  Clergy | Nursing, psychiatric, and ho | Probation officers and corre | Health educators & community
#11 Social worker -> Social workers
   title only               Counselors | Social and human service ass | Clergy | Health educators & community
   + skill answers          Counselors | Social and human service ass | Clergy | Health educators & community
   + description            Social and human service ass | Clergy | Health educators & community | Probation officers and corre
   answers only (no title)  Security guards and gaming s | Driver/sales workers and tr

   title only               Social workers | Social and human service ass | Clergy | Probation officers and corre
   + skill answers          Social workers | Social and human service ass | Clergy | Probation officers and corre
   + description            Social and human service ass | Clergy | Probation officers and corre | Service sales representative
   answers only (no title)  Registered nurses | Social and human service ass | Physicians and surgeons | Probation officers and corre
#13 Lawyer -> Lawyers
   title only               Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Other teachers and instructo
   + skill answers          Physicians and surgeons | Judicial law clerks | Paralegals and legal assista | Compliance, project & other 
   + description            Physicians and surgeons | Paralegals and legal assista | Compliance, project & other  | Other teachers and instructo
   answers only (no title)  Remote sensing & other physi | Psychologist

   title only               Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Other teachers and instructo
   + skill answers          Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Paralegals and legal assista
   + description            Physicians and surgeons | Paralegals and legal assista | Elementary and middle school | Management analysts
   answers only (no title)  Compliance, project & other  | Accountants and auditors | Chief executives | Management analysts
#15 Social worker -> Social workers
   title only               Counselors | Social and human service ass | Clergy | Health educators & community
   + skill answers          Counselors | Social and human service ass | Clergy | Health educators & community
   + description            Social and human service ass | Clergy | Health educators & community | Probation officers and corre
   answers only (no title)  Clergy | Registered nurses | Health educators & communi

   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Environmental scientists and | Medical scientists
   + description            Preschool and kindergarten t | Psychologists | Urban and regional planners | Lawyers
   answers only (no title)  Software developers, applica | Elementary and middle school | CNC operators & other produc | Designers
#17 Web developer -> Web developers
   title only               Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   + skill answers          Software developers, applica | Computer support specialists | IT project managers, QA test | Computer systems analysts
   + description            Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   answers only (no t

## Goals: what do "more pay", "growing field" and "less AI" actually do?

In the app each goal multiplies "Next steps" by **0.6 + 0.8 × signal**, so the best job on that axis gets ×1.4 and the worst ×0.6. The signals:

| Goal | Signal | Note |
|---|---|---|
| `pay` | percentile of US median wage among all 517 jobs | global, not relative to your job |
| `stability` ("growing field") | percentile of BLS projected growth 2025-35 | global |
| `less_ai` | 1 − percentile of *theoretical* AI exposure (Eloundou) within the job zone | not the BLS tier the stop page shows |

Two things to judge: does a goal move the list enough to matter, and what does it cost in reach (the validated part)? A goal that pushes rare moves to the top is trading evidence for a wish.

Approaches, each re-ranking the same base score (reach × interest nudge, no goals, no experience):
- **no goal**: the base list
- **app (0.6-1.4)**: what ships now
- **strong (0.3-1.7)**: same idea, twice the spread
- **better than now**: no re-weighting; only keep stops that beat your current job on that axis (more pay than now, faster growth than now, lower AI exposure than now), in base order. "More pay" then means more than *you* make, which is what people usually mean.

In [8]:
SIGNALS = {
    'pay': lambda f: f['median_wage'].rank(pct=True),
    'stability': lambda f: f['growth_pct'].rank(pct=True),
    'less_ai': lambda f: 1 - f['ai_theoretical_pct_in_zone'],
}
BETTER_THAN_NOW = {   # (column, True if higher is better)
    'pay': ('median_wage', True),
    'stability': ('growth_pct', True),
    'less_ai': ('ai_theoretical', False),
}


_base_cache = {}


def base(profile):
    """Every stop with the base score: the app's 'Next steps' without goals and experience (cached per profile)."""
    key = repr(replace(profile, goals=[], experience_years=None))
    if key not in _base_cache:
        rec = engine.recommend(replace(profile, goals=[], experience_years=None), sort='best', limit=600)
        _base_cache[key] = (pd.DataFrame(rec['results']).set_index('census'), rec['origin'])
    return _base_cache[key]


def no_goal(frame, origin, goal):
    return frame['best']


def multiplier(low, high):
    def score(frame, origin, goal):
        return frame['best'] * (low + (high - low) * SIGNALS[goal](frame)).fillna(1.0)
    return score


def better_than_now(frame, origin, goal):
    column, higher = BETTER_THAN_NOW[goal]
    now = origin.get(column) if origin else None
    if now is None or pd.isna(now):
        return frame['best']
    better = frame[column] > now if higher else frame[column] < now
    return frame['best'].where(better)   # NaN = dropped


GOAL_APPROACHES = {'no goal': no_goal, 'app (0.6-1.4)': multiplier(0.6, 1.4),
                   'strong (0.3-1.7)': multiplier(0.3, 1.7), 'better than now': better_than_now}


def top(profile, goal, approach, n=8):
    frame, origin = base(profile)
    score = GOAL_APPROACHES[approach](frame, origin, goal)
    return frame.assign(score=score).dropna(subset=['score']).sort_values('score', ascending=False).head(n)


tim_plain = replace(tim_cv, goals=[])
print('Tim now:', base(tim_plain)[1]['title'], '| goals as saved:', tim_cv.goals)
pd.DataFrame({name: top(tim_plain, 'pay', name)['title'].str[:34].reset_index(drop=True) for name in GOAL_APPROACHES})

Tim now: Computer and information research scientists | goals as saved: ['passion', 'less_ai']


,no goal,app (0.6-1.4),strong (0.3-1.7),better than now
0,Computer systems analysts,Computer systems analysts,Computer systems analysts,Lawyers
1,Computer programmers,Computer programmers,Computer programmers,Physicians and surgeons
2,Web developers,Web developers,Web developers,Computer and information systems m
3,Computer support specialists,Computer support specialists,Computer support specialists,Architectural and engineering mana
4,Information security analysts,Information security analysts,Information security analysts,Chief executives
5,Operations research analysts,Computer network architects,Computer network architects,Computer hardware engineers
6,Computer network architects,Operations research analysts,Operations research analysts,Marketing and sales managers
7,Network and computer systems admin,Network and computer systems admin,Network and computer systems admin,Financial managers


### Tim, each goal × approach

The first table is "more pay". Below: for every goal and approach, what the top 8 look like on average. `kept` = how many of the no-goal top 8 survive; `reach rank` = mean reach rank of the top 8 (higher = rarer moves, i.e. what the goal costs).

In [9]:
def summary(profile, n=8):
    frame, origin = base(profile)
    plain = set(top(profile, 'pay', 'no goal', n).index)
    rows = []
    for goal in SIGNALS:
        for name in GOAL_APPROACHES:
            t = top(profile, goal, name, n)
            rows.append({'goal': goal, 'approach': name, 'kept': len(plain & set(t.index)),
                         'median pay': t['median_wage'].median(), 'growth %': t['growth_pct'].mean(),
                         'AI theory pct': t['ai_theoretical_pct_in_zone'].mean(),
                         'reach rank': t['reach_rank'].mean(), 'n': len(t)})
    return pd.DataFrame(rows).set_index(['goal', 'approach']).round(2)


print('Tim now: pay', base(tim_plain)[1]['median_wage'], '| growth', base(tim_plain)[1]['growth_pct'])
summary(tim_plain)

Tim now: pay 140300.0 | growth 21.7617


kept  median pay  growth %  AI theory pct  \
goal      approach                                                      
pay       no goal              8    99941.50      4.94           0.90   
          app (0.6-1.4)        8    99941.50      4.94           0.90   
          strong (0.3-1.7)     8    99941.50      4.94           0.90   
          better than now      0   168920.00      7.11           0.50   
stability no goal              8    99941.50      4.94           0.90   
          app (0.6-1.4)        8    99941.50      4.94           0.90   
          strong (0.3-1.7)     7   103120.00      6.17           0.83   
          better than now      0   119675.78     33.08           0.46   
less_ai   no goal              8    99941.50      4.94           0.90   
          app (0.6-1.4)        8    99941.50      4.94           0.90   
          strong (0.3-1.7)     7   103120.00      6.17           0.83   
          better than now      3   114306.53      6.32           0.54   

                            reach rank  n  
goal      approach                         
pay       no goal                 4.50  8  
          app (0.6-1.4)           4.50  8  
          strong (0.3-1.7)        4.50  8  
          better than now        23.75  8  
stability no goal                 4.50  8  
          app (0.6-1.4)           4.50  8  
          strong (0.3-1.7)        4.75  8  
          better than now       320.40  5  
less_ai   no goal                 4.50  8  
          app (0.6-1.4)           4.50  8  
          strong (0.3-1.7)        4.75  8  
          better than now         8.12  8

In [10]:
for goal in ['stability', 'less_ai']:
    print(goal)
    display(pd.DataFrame({name: top(tim_plain, goal, name)['title'].str[:34].reset_index(drop=True) for name in GOAL_APPROACHES}))

stability


,no goal,app (0.6-1.4),strong (0.3-1.7),better than now
0,Computer systems analysts,Computer systems analysts,Computer systems analysts,Medical and health services manage
1,Computer programmers,Web developers,Web developers,Nurse practitioners
2,Web developers,Computer programmers,Information security analysts,Wind turbine service technicians
3,Computer support specialists,Computer support specialists,Operations research analysts,Solar photovoltaic installers
4,Information security analysts,Information security analysts,Computer network architects,Data scientists & other math roles
5,Operations research analysts,Operations research analysts,Computer support specialists,NaN
6,Computer network architects,Computer network architects,Computer programmers,NaN
7,Network and computer systems admin,Network and computer systems admin,"Compliance, project & other manage",NaN


less_ai


,no goal,app (0.6-1.4),strong (0.3-1.7),better than now
0,Computer systems analysts,Computer systems analysts,Computer systems analysts,Computer systems analysts
1,Computer programmers,Computer programmers,Computer support specialists,Computer support specialists
2,Web developers,Computer support specialists,Computer programmers,Operations research analysts
3,Computer support specialists,Web developers,Web developers,"Compliance, project & other manage"
4,Information security analysts,Information security analysts,Information security analysts,Lawyers
5,Operations research analysts,Computer network architects,Computer network architects,Physicians and surgeons
6,Computer network architects,Operations research analysts,"Compliance, project & other manage",Actuaries
7,Network and computer systems admin,Network and computer systems admin,Operations research analysts,Management analysts


### The 17 respondents, averaged

Same summary for every survey person (title + description, no goals), averaged. Shows whether the pattern from Tim's profile holds.

In [11]:
survey_profiles = [Profile(job=p['job'], description=p['description']) for p in people]
everyone = pd.concat([summary(p) for p in survey_profiles], keys=range(len(survey_profiles)))
everyone.groupby(level=['goal', 'approach']).mean().round(2)

kept  median pay  growth %  AI theory pct  \
goal      approach                                                      
less_ai   app (0.6-1.4)     7.29    92422.33      3.96           0.51   
          better than now   4.76    86457.91      6.13           0.50   
          no goal           8.00    91444.09      4.29           0.56   
          strong (0.3-1.7)  6.76    95367.75      4.48           0.48   
pay       app (0.6-1.4)     7.59    94549.25      4.43           0.56   
          better than now   3.59   128915.57      6.08           0.52   
          no goal           8.00    91444.09      4.29           0.56   
          strong (0.3-1.7)  7.29    96893.35      4.57           0.56   
stability app (0.6-1.4)     7.47    92418.31      4.98           0.56   
          better than now   3.71   101283.61     11.64           0.57   
          no goal           8.00    91444.09      4.29           0.56   
          strong (0.3-1.7)  6.94    94550.30      6.59           0.55   

                            reach rank     n  
goal      approach                            
less_ai   app (0.6-1.4)           4.77  8.00  
          better than now        10.26  8.00  
          no goal                 4.50  8.00  
          strong (0.3-1.7)        5.17  8.00  
pay       app (0.6-1.4)           4.60  8.00  
          better than now        24.65  7.82  
          no goal                 4.50  8.00  
          strong (0.3-1.7)        4.90  8.00  
stability app (0.6-1.4)           4.63  8.00  
          better than now        16.38  8.00  
          no goal                 4.50  8.00  
          strong (0.3-1.7)        5.10  8.00

## Experience: what does the ×1.2 / ×0.7 do?

BLS lists the related experience a job usually needs: none (437 jobs), under 5 years (55), or 5+ years (24, mostly management). With 5+ years, the 24 jobs get ×1.2; with less, ×0.7. Everything else is untouched. Here: Tim's list at different years, and how many of the top 10 need 5+ years.

In [12]:
def with_years(profile, years, n=10):
    return engine.recommend(replace(profile, goals=[], experience_years=years), sort='best', limit=n)['results']


YEARS = [None, 1, 3, 7, 12]
pd.DataFrame({str(y): [r['title'][:30] + (' *' if r['work_experience'] == '5 years or more' else '') for r in with_years(tim_plain, y)]
              for y in YEARS})

,None,1,3,7,12
0,Computer systems analysts,Computer systems analysts,Computer systems analysts,Computer systems analysts,Computer systems analysts
1,Computer programmers,Computer programmers,Computer programmers,Computer programmers,Computer programmers
2,Web developers,Web developers,Web developers,Web developers,Web developers
3,Computer support specialists,Computer support specialists,Computer support specialists,Computer support specialists,Computer support specialists
4,Information security analysts,Information security analysts,Information security analysts,Computer network architects *,Computer network architects *
5,Operations research analysts,Operations research analysts,Operations research analysts,Information security analysts,Information security analysts
6,Computer network architects *,Network and computer systems a,Network and computer systems a,Operations research analysts,Operations research analysts
7,Network and computer systems a,Computer network architects *,Computer network architects *,Network and computer systems a,Network and computer systems a
8,"Compliance, project & other ma","Compliance, project & other ma","Compliance, project & other ma","Compliance, project & other ma","Compliance, project & other ma"
9,Lawyers,Lawyers,Lawyers,Lawyers,Lawyers


In [13]:
def senior_count(profile, years, n=10):
    return sum(r['work_experience'] == '5 years or more' for r in with_years(profile, years, n))


pd.DataFrame({'Tim': {str(y): senior_count(tim_plain, y) for y in YEARS},
              'survey mean': {str(y): np.mean([senior_count(p, y) for p in survey_profiles]) for y in YEARS}}).T

,None,1,3,7,12
Tim,1.000000,1.000000,1.000000,1.000000,1.000000
survey mean,0.764706,0.352941,0.352941,1.058824,1.058824


`*` = usually needs 5+ years of related experience. Things to check: does a junior person still see management at all (should they)? Does 12 years push management in too hard? The multiplier ignores *which* field the experience is in: 7 years as a nurse counts the same for "IT manager".

## Findings

TODO (Tim): which outcomes look right, which look wrong, and why.

Goals (TODO): does the app's multiplier move the list enough to matter? Is "better than now" closer to what people mean by "more pay"? What reach do we give up for each goal? Should `less_ai` use the BLS tier the stop page shows?

Experience (TODO): keep the ×1.2 / ×0.7, soften it, or only show the note on the stop page?